In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4.1-nano")
output_parser = StrOutputParser()


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
# os.getenv('OPENAI_API_KEY')

In [ ]:
llm.invoke("Where is the capital of Korea? Answer me in Korean")

# 1. 나라 -> 음식 추천

In [ ]:
food_prompt = PromptTemplate(
    template="""
    Please recommend the most famous food from {country}.    
    When answering, write full name of country and,
    please use the national flag and food of the answer as an emoji.
    And give a short explanation of the name of the food and one line.

    If the user wants a specific format, follow this instruction: {format}
    """,
    input_variables=["country", "format"]
)


food_chain = food_prompt | llm | output_parser

output_parser.invoke(llm.invoke(food_prompt.invoke({"country":"Japan", "format": "Just the food name only"})))

# 2. 음식 -> 레시피

In [ ]:
recipe_prompt = PromptTemplate(
    template="""
    Write a detailed recipe for the food: {food}.
    Include ingredients and cooking instructions.
    When answering, please use the national flag and food of the answer as an emoji.

    If the user wants a specific output style, follow this instruction: {style}
    """,
    input_variables=["food", "style"]
)


recipe_chain = recipe_prompt | llm | output_parser
output_parser.invoke(llm.invoke(recipe_prompt.invoke({"food":"kimchi", "style": "use emoji of food"})))

# 3. 체인 연결

In [ ]:
from langchain_core.runnables import RunnablePassthrough
final_chain = {"country": RunnablePassthrough(), "format": RunnablePassthrough()} | {"food": food_chain
} | {"food": RunnablePassthrough(), "style": lambda x: x["format"] } | recipe_chain


# 4. 출력

In [ ]:
result = final_chain.invoke({
    "country": "Japan",
    "format": "한국어로 답변해봐"
})
print(result)
